In [1]:
import kagglehub
kagglehub.login()

In [2]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [3]:
import unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
reference_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "barnobarno/aimo-training-dataset",
    "reference.csv",
)
reference_df.head()

,id,problem,answer
0,0e644e,Let $ABC$ be an acute-angled triangle with int...,336
1,26de63,Define a function $f \colon \mathbb{Z}_{\geq 1...,32951
2,424e18,A tournament is held with $2^{20}$ runners eac...,21818
3,42d360,"On a blackboard, Ken starts off by writing a p...",32193
4,641659,"Let $ABC$ be a triangle with $AB \neq AC$, cir...",57447


In [5]:
import os 
lora_path = kagglehub.dataset_download("barnobarno/aimo-oss-finetuned-20b-lora")
os.listdir(lora_path)

['adapter_model.safetensors',
 'special_tokens_map.json',
 'adapter_config.json',
 'tokenizer_config.json',
 'chat_template.jinja',
 'README.md',
 'tokenizer.json']

In [6]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.80G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.17G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [7]:
from transformers import TextStreamer

messages = [
    {"role": "user", "content": "Solve x^5 + 3x^4 - 10 = 3."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
    reasoning_effort = "medium", # **NEW!** Set reasoning effort to low, medium or high
).to("cuda")

_ = model.generate(**inputs, max_new_tokens = 1024, streamer = TextStreamer(tokenizer))

<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.
Knowledge cutoff: 2024-06
Current date: 2025-12-31

Reasoning: medium

# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>user<|message|>Solve x^5 + 3x^4 - 10 = 3.<|end|><|start|>assistant<|channel|>analysis<|message|>The problem: "Solve x^5 + 3x^4 - 10 = 3." So equation: x^5 + 3x^4 - 10 = 3. Bring all to one side: x^5 + 3x^4 - 13 = 0. Solve polynomial equation? It's quintic. Might factor or find rational roots. Try integer roots: factors of 13: ±1, ±13. Trying x=1: 1+3-13=-9. x= -1: -1+3-13=-11. x=13: huge. x=-13 huge. So no integer root. Maybe rational root p divides 13, q divides leading coefficient 1: ±1, ±13. Already tried: no. So no rational root. Perhaps factor of form (x^2 + ax + b)(x^3 + cx^2 + dx + e)? Possibly but unlikely.

But maybe equation meant x^5/ + 3x^4 -10 = 3? Wait could be misprint: maybe it's x^(5 + 3x^4) -10=3? No. Actually 

In [8]:
import re
import time
import pandas as pd
from transformers import TextStreamer

SYSTEM_PROMPT = """You are an expert math problem solver. Solve the given problem step by step.
At the end of your solution, you MUST provide your final answer as an integer inside a box using the format:
\\boxed{INTEGER}

Important:
- The final answer must be an INTEGER (whole number)
- Always use \\boxed{} for your final answer
- Only one \\boxed{} should appear at the end with your final numerical answer"""

def extract_answer(text):
    """Extract the final integer answer from model output (expects \\boxed{INTEGER} format)."""
    # Find all boxed answers and take the last one (final answer)
    boxed_matches = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed_matches:
        last_boxed = boxed_matches[-1].strip()
        # Remove any commas, spaces, or formatting
        cleaned = re.sub(r'[,\s]', '', last_boxed)
        try:
            # Handle negative numbers and floats that should be integers
            return int(float(cleaned))
        except:
            pass
    
    # Fallback: try to find answer after common patterns
    patterns = [
        r'[Ff]inal [Aa]nswer[:\s]+[-]?(\d+)',
        r'[Aa]nswer[:\s]+[-]?(\d+)',
        r'[Tt]he answer is[:\s]+[-]?(\d+)',
        r'= ([-]?\d+)\s*$',
    ]
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            try:
                return int(match.group(1))
            except:
                pass
    return None

def run_inference(model, tokenizer, problem, max_tokens=2048, reasoning_effort="medium"):
    """Run inference on a single problem and return results."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{problem}"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        reasoning_effort=reasoning_effort,
    ).to("cuda")
    
    input_tokens = inputs['input_ids'].shape[1]
    
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=max_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )
    end_time = time.time()
    
    output_tokens = outputs.shape[1] - input_tokens
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = tokenizer.decode(outputs[0][input_tokens:], skip_special_tokens=True)
    
    return {
        'response': generated_text,
        'input_tokens': input_tokens,
        'output_tokens': output_tokens,
        'total_tokens': outputs.shape[1],
        'inference_time': end_time - start_time,
    }

def evaluate_model(model, tokenizer, df, n_samples=4, model_name="Model"):
    """Evaluate model on first n samples of dataframe."""
    results = []
    
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")
    
    for idx in range(min(n_samples, len(df))):
        row = df.iloc[idx]
        problem = row['problem']
        correct_answer = int(row['answer'])
        
        print(f"\n--- Problem {idx+1} ---")
        print(f"Problem: {problem[:100]}...")
        print(f"Correct Answer: {correct_answer}")
        
        result = run_inference(model, tokenizer, problem)
        predicted_answer = extract_answer(result['response'])
        is_correct = predicted_answer == correct_answer
        
        print(f"Predicted Answer: {predicted_answer}")
        print(f"Correct: {'✓' if is_correct else '✗'}")
        print(f"Tokens: {result['input_tokens']} in / {result['output_tokens']} out")
        print(f"Time: {result['inference_time']:.2f}s")
        
        results.append({
            'problem_idx': idx,
            'correct_answer': correct_answer,
            'predicted_answer': predicted_answer,
            'is_correct': is_correct,
            'input_tokens': result['input_tokens'],
            'output_tokens': result['output_tokens'],
            'total_tokens': result['total_tokens'],
            'inference_time': result['inference_time'],
            'response': result['response']
        })
    
    return pd.DataFrame(results)

# Test samples
test_df = reference_df.head(4)
print(f"Testing on {len(test_df)} problems")
print(f"\nSystem Prompt being used:")
print("-" * 40)
print(SYSTEM_PROMPT)
print("-" * 40)
print(f"\nProblems preview:")
print(test_df[['problem', 'answer']].head())

Testing on 4 problems

System Prompt being used:
----------------------------------------
You are an expert math problem solver. Solve the given problem step by step.
At the end of your solution, you MUST provide your final answer as an integer inside a box using the format:
\boxed{INTEGER}

Important:
- The final answer must be an INTEGER (whole number)
- Always use \boxed{} for your final answer
- Only one \boxed{} should appear at the end with your final numerical answer
----------------------------------------

Problems preview:
                                             problem  answer
0  Let $ABC$ be an acute-angled triangle with int...     336
1  Define a function $f \colon \mathbb{Z}_{\geq 1...   32951
2  A tournament is held with $2^{20}$ runners eac...   21818
3  On a blackboard, Ken starts off by writing a p...   32193


In [9]:
# ============================================
# TEST 1: Model WITHOUT Adapters (Base Model)
# ============================================
print("\n" + "="*70)
print("PHASE 1: EVALUATING BASE MODEL (WITHOUT ADAPTERS)")
print("="*70)

FastLanguageModel.for_inference(model)  # Enable native inference
results_without_adapter = evaluate_model(model, tokenizer, test_df, n_samples=4, model_name="Base Model (No Adapter)")


PHASE 1: EVALUATING BASE MODEL (WITHOUT ADAPTERS)

Evaluating: Base Model (No Adapter)

--- Problem 1 ---
Problem: Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie ...
Correct Answer: 336
Predicted Answer: None
Correct: ✗
Tokens: 300 in / 2048 out
Time: 192.43s

--- Problem 2 ---
Problem: Define a function $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ by
\begin{equation*}
    f(...
Correct Answer: 32951


KeyboardInterrupt: 

In [ ]:
# ============================================
# Load LoRA Adapters
# ============================================
print("\n" + "="*70)
print("LOADING LORA ADAPTERS")
print("="*70)

model.load_adapter(lora_path, adapter_name="aimo_lora")
model.set_adapter("aimo_lora")
print(f"Loaded adapter from: {lora_path}")

In [ ]:
# ============================================
# TEST 2: Model WITH Adapters (Fine-tuned)
# ============================================
print("\n" + "="*70)
print("PHASE 2: EVALUATING MODEL WITH LORA ADAPTERS")
print("="*70)

FastLanguageModel.for_inference(model)  # Enable native inference
results_with_adapter = evaluate_model(model, tokenizer, test_df, n_samples=4, model_name="Fine-tuned Model (With Adapter)")

In [ ]:
# ============================================
# COMPARISON: With Adapter vs Without Adapter
# ============================================
print("\n" + "="*70)
print("COMPARISON SUMMARY: BASE MODEL vs FINE-TUNED MODEL")
print("="*70)

# Calculate metrics for both
def calculate_metrics(results_df, name):
    accuracy = results_df['is_correct'].mean() * 100
    avg_output_tokens = results_df['output_tokens'].mean()
    avg_total_tokens = results_df['total_tokens'].mean()
    avg_time = results_df['inference_time'].mean()
    total_correct = results_df['is_correct'].sum()
    total_problems = len(results_df)
    
    return {
        'Model': name,
        'Accuracy (%)': accuracy,
        'Correct/Total': f"{total_correct}/{total_problems}",
        'Avg Output Tokens': avg_output_tokens,
        'Avg Total Tokens': avg_total_tokens,
        'Avg Inference Time (s)': avg_time,
        'Total Time (s)': results_df['inference_time'].sum()
    }

metrics_without = calculate_metrics(results_without_adapter, "Base Model (No Adapter)")
metrics_with = calculate_metrics(results_with_adapter, "Fine-tuned (With Adapter)")

comparison_df = pd.DataFrame([metrics_without, metrics_with])
print("\n📊 Overall Metrics Comparison:")
print(comparison_df.to_string(index=False))

# Detailed per-problem comparison
print("\n\n📋 Per-Problem Comparison:")
print("-" * 70)

detailed_comparison = pd.DataFrame({
    'Problem': range(1, len(test_df) + 1),
    'Correct Answer': results_without_adapter['correct_answer'].values,
    'Base Pred': results_without_adapter['predicted_answer'].values,
    'Base Correct': results_without_adapter['is_correct'].apply(lambda x: '✓' if x else '✗').values,
    'FT Pred': results_with_adapter['predicted_answer'].values,
    'FT Correct': results_with_adapter['is_correct'].apply(lambda x: '✓' if x else '✗').values,
    'Base Tokens': results_without_adapter['output_tokens'].values,
    'FT Tokens': results_with_adapter['output_tokens'].values,
})
print(detailed_comparison.to_string(index=False))

# Improvement analysis
print("\n\n📈 Improvement Analysis:")
print("-" * 70)
accuracy_diff = metrics_with['Accuracy (%)'] - metrics_without['Accuracy (%)']
token_diff = metrics_with['Avg Output Tokens'] - metrics_without['Avg Output Tokens']
time_diff = metrics_with['Avg Inference Time (s)'] - metrics_without['Avg Inference Time (s)']

print(f"Accuracy Change: {'+' if accuracy_diff >= 0 else ''}{accuracy_diff:.1f}%")
print(f"Token Count Change: {'+' if token_diff >= 0 else ''}{token_diff:.0f} tokens (avg)")
print(f"Inference Time Change: {'+' if time_diff >= 0 else ''}{time_diff:.2f}s (avg)")

if accuracy_diff > 0:
    print(f"\n✅ Fine-tuned model improved accuracy by {accuracy_diff:.1f}%")
elif accuracy_diff < 0:
    print(f"\n⚠️ Fine-tuned model decreased accuracy by {abs(accuracy_diff):.1f}%")
else:
    print(f"\n➡️ No change in accuracy between models")